In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import ShuffleSplit
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# NLTK Kneser-Ney interpolated (Hindle et al. ICSE 2012)
from nltk.lm import KneserNeyInterpolated, Laplace
from nltk.lm.preprocessing import padded_everygram_pipeline, pad_sequence
from nltk.lm.vocabulary import Vocabulary
from nltk.util import ngrams

# Same filtering as a.ipynb
df = pd.read_csv("../../dataset/data/final_dataset_new_corrected.csv")
for col in ["doc_entropy", "doc_code_overlap", "doc_redundancy"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna(subset=["doc_entropy", "doc_code_overlap", "doc_redundancy"])

agent_docs     = df[df["group"] == "agent"]["doc_text"].astype(str).tolist()
developer_docs = df[df["group"] == "human"]["doc_text"].astype(str).tolist()

agent_docs_gr10 = [doc for doc in agent_docs if len(doc.split()) >= 10]
developer_docs_gr10 = [doc for doc in developer_docs if len(doc.split()) >= 10]

print(f"Agent docs:     {len(agent_docs)}")
print(f"Developer docs: {len(developer_docs)}")
print(f"Agent docs >= 10 words:     {len(agent_docs_gr10)}")
print(f"Developer docs >= 10 words: {len(developer_docs_gr10)}")

Agent docs:     3744
Developer docs: 2177
Agent docs >= 10 words:     1966
Developer docs >= 10 words: 1259


In [2]:
# Words appearing fewer than UNK_CUTOFF times in training are replaced with <UNK>.
# This ensures <UNK> has a non-zero count in training so test OOV words get
# a finite probability rather than zero (which would make entropy = inf).
UNK       = '<UNK>'
UNK_CUTOFF = 2


def train_lm(n, docs):
    """
    Train an order-n language model on whitespace-tokenized docs.
    Uses Kneser-Ney interpolated smoothing for n >= 2 (Hindle et al.);
    falls back to Laplace for n=1 since KN requires bigram context counts.
    """
    tokenized = [d.split() for d in docs if d.split()]

    # Build a vocabulary; mark words below cutoff as <UNK>
    all_words  = [w for sent in tokenized for w in sent]
    train_vocab = Vocabulary(all_words, unk_cutoff=UNK_CUTOFF)

    # Replace rare words with <UNK> in training sequences so the model
    # assigns a non-zero probability to <UNK> at test time
    tokenized_unk = [
        [w if w in train_vocab else UNK for w in sent]
        for sent in tokenized
    ]

    train_data, fitted_vocab = padded_everygram_pipeline(n, tokenized_unk)
    model = KneserNeyInterpolated(n) if n > 1 else Laplace(n)
    model.fit(train_data, fitted_vocab)
    return model, train_vocab


def doc_cross_entropy(model, train_vocab, doc, n):
    """
    Per-sample cross-entropy (bits) of a single doc under the given model.
    Matches Hindle et al.: log2, padded n-grams, mean over n-gram positions.
    """
    tokens = doc.split()
    if not tokens:
        return np.nan
    # Map OOV test tokens to <UNK> using the training vocabulary
    tokens = [w if w in train_vocab else UNK for w in tokens]
    padded = list(pad_sequence(
        tokens, n,
        pad_left=True,  left_pad_symbol='<s>',
        pad_right=True, right_pad_symbol='</s>',
    ))
    test_ngrams = list(ngrams(padded, n))
    if not test_ngrams:
        return np.nan
    try:
        ce = model.entropy(test_ngrams)   # returns bits (log2)
        return ce if np.isfinite(ce) else np.nan
    except (ZeroDivisionError, ValueError):
        return np.nan

In [ ]:
# the following cell is used to train our model with 100 random sample validation and compute the 
# cross-entropy for each document in the test set. this cell uses our whole dataset
N_SPLITS = 100
records  = []


for n in range(1, 6):
    print(f"n={n}", end=" ", flush=True)
    for label, docs in [("Agent", agent_docs), ("Developer", developer_docs)]:
        arr = np.array(docs, dtype=object)
        ss = ShuffleSplit(n_splits=N_SPLITS, test_size=0.1, random_state=42) # split the data
        for train_idx, test_idx in ss.split(arr):
            model, train_vocab = train_lm(n, arr[train_idx])
            for doc in arr[test_idx]:
                ce = doc_cross_entropy(model, train_vocab, doc, n)
                if not np.isnan(ce):
                    records.append({"n": n, "Group": label, "cross_entropy": ce})
        print(".", end="", flush=True)
    print(" done")

results = pd.DataFrame(records)
print(f"\nTotal records: {len(results)}")
print(results.groupby(["n", "Group"]).size().unstack())

results.to_csv("n_gram_cross_entropy.csv", index=False)

n=1 .. done
n=2 .. done
n=3 .. done
n=4 .. done
n=5 .. done

Total records: 29605
Group  Agent  Developer
n                      
1       3744       2177
2       3744       2177
3       3744       2177
4       3744       2177
5       3744       2177


In [ ]:
# # the following cell is used to train our model with 10 random sample cross validation, but uses all samples 
# # with words >= 10
# # the following cell is used to train our model with 10-fold cross validation and compute the 
# # cross-entropy for each document in the test set. this cell uses our whole dataset
# N_SPLITS = 10
# records_gr10      = []

# for n in range(1, 11):
#     print(f"n={n}", end=" ", flush=True)
#     for label, docs in [("Agent", agent_docs_gr10), ("Developer", developer_docs_gr10)]:
#         arr = np.array(docs, dtype=object)
#         kf  = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
#         for train_idx, test_idx in kf.split(arr):
#             model, train_vocab = train_lm(n, arr[train_idx])
#             for doc in arr[test_idx]:
#                 ce = doc_cross_entropy(model, train_vocab, doc, n)
#                 if not np.isnan(ce):
#                     records.append({"n": n, "Group": label, "cross_entropy": ce})
#         print(".", end="", flush=True)
#     print(" done")

# results = pd.DataFrame(records)
# print(f"\nTotal records: {len(results)}")
# print(results.groupby(["n", "Group"]).size().unstack())


In [4]:
summary = (
    results.groupby(["n", "Group"])["cross_entropy"]
    .agg(mean="mean", std="std")
    .round(4)
)
print("Mean and Std of per-sample cross-entropy (bits) — Kneser-Ney interpolated")
print("=" * 60)
print(summary.to_string())

Mean and Std of per-sample cross-entropy (bits) — Kneser-Ney interpolated
               mean     std
n Group                    
1 Agent      9.3093  1.7027
  Developer  8.6663  1.8493
2 Agent      7.1549  2.3636
  Developer  6.2678  2.5195
3 Agent      6.6395  3.0308
  Developer  5.5471  3.3581
4 Agent      6.3922  3.0792
  Developer  5.3294  3.5203
5 Agent      6.1992  3.0029
  Developer  5.1786  3.4878


In [ ]:
def sig_label(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

def rank_biserial(x, y, U=None):
    """Rank-biserial correlation effect size for Mann-Whitney U (Wendt, 1972)."""
    if U is None:
        U, _ = mannwhitneyu(x, y, alternative="two-sided")
    return 1 - (2 * U) / (len(x) * len(y))

palette = {"Agent": "#A7C7E7", "Developer": "#BDE5B8"}

fig, ax = plt.subplots(figsize=(11, 6), dpi=150)

sns.boxplot(
    data=results, x="n", y="cross_entropy", hue="Group",
    palette=palette, showfliers=False, linewidth=1.2,
    order=[1, 2, 3, 4, 5], ax=ax,
)

ax.set_xlabel("n-gram order (n)", fontsize=13)
ax.set_ylabel("Per-sample cross-entropy (bits)", fontsize=13)
ax.set_title(
    "N-gram Cross-Entropy: Agent vs. Developer Documentation\n"
    "(Kneser-Ney interpolated, 10-fold CV)",
    fontsize=14, fontweight="bold"
)
ax.legend(title="Group", fontsize=11, title_fontsize=11)
ax.tick_params(axis="both", labelsize=11)
plt.tight_layout()
plt.savefig("ngram_cross_entropy.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nMann-Whitney U p-values (two-sided, uncorrected) and rank-biserial correlation:")
for n in range(1, 6):
    ag = results[(results["n"] == n) & (results["Group"] == "Agent")]["cross_entropy"].values
    dv = results[(results["n"] == n) & (results["Group"] == "Developer")]["cross_entropy"].values
    U, p = mannwhitneyu(ag, dv, alternative="two-sided")
    rbc = rank_biserial(ag, dv, U=U)
    print(f"  n={n}: p={p:.4e}  {sig_label(p)}  rank-biserial r={rbc:+.3f}")